In [1]:
%env HSA_OVERRIDE_GFX_VERSION=10.3.0
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import re
from tqdm.notebook import tqdm

env: HSA_OVERRIDE_GFX_VERSION=10.3.0


In [2]:
def create_atomic_img_database():
    df_pde = pd.read_csv('PubChemElements_all.csv').fillna(0)
    useCols = ["AtomicNumber","Electronegativity","AtomicRadius","IonizationEnergy","ElectronAffinity","MeltingPoint","BoilingPoint","Density"]
    colMaxes = [df_pde[prop_name].abs().max() for prop_name in useCols]
    # print(colMaxes)
    
    atomic_img_database = np.zeros((df_pde["AtomicNumber"].max()+1, len(useCols), len(useCols)))
    for atomic_num in df_pde["AtomicNumber"]:
        for j, prop_name in enumerate(useCols):
            prop = df_pde.loc[df_pde['AtomicNumber'] == atomic_num][prop_name].item()
            atomic_img_database[atomic_num,0,j] = prop/colMaxes[j]
            # atomic_img_database[atomic_num,0,j] = prop
        for k in range(1, len(useCols)):
            atomic_img_database[atomic_num,k] = np.roll(atomic_img_database[atomic_num,k-1], 1)
        # plt.imshow(atomic_img_database[atomic_num-1])
        # plt.show()
    return atomic_img_database, colMaxes

def formula_to_molecular_img(formula, atomic_img_db):
    df_pde = pd.read_csv('PubChemElements_all.csv', usecols=['AtomicNumber', 'Symbol'])
    matches = re.findall(r'([A-Z][a-z]*)(\d*)', formula)
    molecular_img = []
    for element, count in matches:
        # If a number exists, use it; otherwise default to 1
        # 'V2' -> count is '2', 'Sc' -> count is '' (which becomes 1)
        n = int(count) if count else 1
        molecular_img.extend([atomic_img_db[df_pde.loc[df_pde['Symbol'] == element]['AtomicNumber'].item()]] * n)
        # formula_extended.extend([element] * n)
        # print(formula_extended)
    if len(molecular_img) < 4:
        molecular_img.append(np.zeros((8,8)))
    # for element in formula_extended:
    #     atomic_num_list.append(element_dict[element])
    return np.array(molecular_img)

In [3]:
df = pd.read_csv('heusler_magnetic.csv', usecols=['formula', 'heusler type', 'num_electron', 'struct type', 'latt const', 'tetragonality', 'e_form', 'pol fermi', 'gap width', 'stability', 'mu_b', 'mu_b saturation'])

heusler_map = ['Full Heusler', 'Half Heusler', 'Inverse Heusler']
for typ in heusler_map:
    df[typ] = (df['heusler type'] == typ)

df['struct type'].replace('Tetragonal', 'tetragonal', inplace=True)
struct_map = {'D022':[1,0,0,0,0,0], 'L21':[0,1,0,0,0,0], 'C1b':[0,0,1,0,0,0], 'tetragonal':[0,0,0,1,0,0], 'triclinic':[0,0,0,0,1,0], 'Xa':[0,0,0,0,0,1]}
df['struct type'] = df['struct type'].map(struct_map)

gap_width_arr = pd.to_numeric(df['gap width'], errors='coerce')
df['gap width'] = gap_width_arr.fillna(0)

pol_fermi = pd.to_numeric(df['pol fermi'], errors='coerce')
df['pol fermi'] = pol_fermi.fillna(0)

e_form = pd.to_numeric(df['e_form'], errors='coerce')
# df['e_form'] = e_form.fillna(100)

mbs = pd.to_numeric(df['mu_b saturation'], errors='coerce')
mbs[np.isnan(mbs)] = 0
df['mu_b saturation'] = mbs

# with pd.option_context("future.no_silent_downcasting", True):
#     df['stability'] = df['stability'].fillna(False).astype(bool)

# instead of replacing values and poisoning the data, we simply clean it all up
df = df.dropna()
df['stability'] = df['stability'].astype(bool)

atomic_img_db, _ = create_atomic_img_database()
molecular_img_lst = [formula_to_molecular_img(x, atomic_img_db) for x in df['formula']]
e_form_max = df['e_form'].max()
latt_sz_max = df['latt const'].max()
# print(x_train.shape)

x_train_up = np.array(molecular_img_lst, dtype=np.float32)
y_train_eform_up = df['e_form'].to_numpy(dtype=np.float32).reshape((1048,1))/e_form_max
y_train_latsz_up = df['latt const'].to_numpy(dtype=np.float32).reshape((1048,1))/latt_sz_max
y_train_stabi_up = df['stability'].to_numpy(dtype=np.float32).reshape((1048,1))
# y_train_heuty_up = df['heusler_type'].to_numpy(dtype=np.float32).reshape((1081,3))

x_train, x_test, y_train_eform, y_test_eform, y_train_latsz, y_test_latsz, y_train_stabi, y_test_stabi = train_test_split(x_train_up, 
                            y_train_eform_up, 
                            y_train_latsz_up, 
                            y_train_stabi_up, 
                            test_size=0.2, random_state=3, shuffle=True)
# df['formula'] = molecular_img_lst
x_train, x_test, y_train_eform, y_test_eform, y_train_latsz, y_test_latsz, y_train_stabi, y_test_stabi = torch.tensor(x_train, device="cuda"), torch.tensor(x_test, device="cuda"), torch.tensor(y_train_eform, device="cuda"), torch.tensor(y_test_eform, device="cuda"), torch.tensor(y_train_latsz, device="cuda"), torch.tensor(y_test_latsz, device="cuda"), torch.tensor(y_train_stabi, device="cuda"), torch.tensor(y_test_stabi, device="cuda")
# x_train, x_test, y_train_eform, y_test_eform, y_train_latsz, y_test_latsz, y_train_stabi, y_test_stabi = x_train.to(device), x_test.to(device), y_train_eform.to(device), y_test_eform.to(device), y_train_latsz.to(device), y_test_latsz.to(device), y_train_stabi.to(device), y_test_stabi.to(device)

print(x_train.shape, y_train_eform.shape, y_train_stabi.shape, y_train_latsz.shape)
print(x_test.shape, y_test_eform.shape, y_test_stabi.shape, y_test_latsz.shape)
# df.to_csv('processed_heusler_data.csv')
# training_df = df.drop(columns=['stability'], inplace=False)
# predict_arr = df['stability'].to_numpy()
# print(training_df['gap width'].unique())
# print(formula_to_atomic_num_lst)

/tmp/ipykernel_17424/1303567816.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['struct type'].replace('Tetragonal', 'tetragonal', inplace=True)


torch.Size([838, 4, 8, 8]) torch.Size([838, 1]) torch.Size([838, 1]) torch.Size([838, 1])
torch.Size([210, 4, 8, 8]) torch.Size([210, 1]) torch.Size([210, 1]) torch.Size([210, 1])


In [4]:
class MyCNNModel(nn.Module):
    def __init__(self):
        super(MyCNNModel, self).__init__()
        self.convlayer_1 = nn.Conv2d(4, 8, 2)
        self.convlayer_2 = nn.Conv2d(8, 16, 4)
        self.deeplayer_1 = nn.Linear(16*4*4, 256)
        # self.deeplayer_1 = nn.Linear(8*7*7, 256)
        self.deeplayer_2 = nn.Linear(256, 32)
        
        self.deeplayer_3_stability = nn.Linear(32, 1) # stability
        self.deeplayer_3_lattconst = nn.Linear(32, 1) # lattice_const
        self.deeplayer_3_formation = nn.Linear(32, 1) # formation energy

    def forward(self, x):
        x = nn.functional.relu(self.convlayer_1(x))
        x = nn.functional.relu(self.convlayer_2(x))
        x = torch.flatten(x, 1)
        x = nn.functional.relu(self.deeplayer_1(x))
        x = nn.functional.relu(self.deeplayer_2(x))
        
        stability = nn.functional.sigmoid(self.deeplayer_3_stability(x))
        lattconst = nn.functional.tanh(self.deeplayer_3_lattconst(x))
        formation = nn.functional.tanh(self.deeplayer_3_formation(x))
        # x = self.deeplayer_3(x)
        # return stability
        return stability, lattconst, formation

device = torch.device("cuda")
my_cnn_net = MyCNNModel()
my_cnn_net.to(device)
loss_fn_stab = nn.BCELoss()
loss_fn_latt = nn.MSELoss()
loss_fn_form = nn.MSELoss()
optimizer_fn = torch.optim.AdamW(my_cnn_net.parameters(), lr=0.001, weight_decay=1e-4)

# Training loop
train_epochs = 5000
min_validation_loss = 1

for epoch in range(train_epochs):
    my_cnn_net.train()
    optimizer_fn.zero_grad()
    # Forward pass
    # output_stability = my_cnn_net(x_train)
    output_stability, output_lattconst, output_formation = my_cnn_net(x_train)
    
    loss_stab = loss_fn_stab(output_stability, y_train_stabi)
    loss_latt = loss_fn_latt(output_lattconst, y_train_latsz)
    loss_form = loss_fn_form(output_formation, y_train_eform)

    # total_loss_val = (loss_stab)
    total_loss_val = (0.2*loss_stab+0.6*loss_latt+1.5*loss_form)
    
    # Backward pass and optimization
    total_loss_val.backward()
    optimizer_fn.step()

    my_cnn_net.eval()
    with torch.no_grad():
        # raw_stab = my_cnn_net(x_test)
        raw_stab, raw_latt, raw_form = my_cnn_net(x_test)
        # total_validation_loss = loss_fn_stab(raw_stab, y_test_stabi)
        total_validation_loss = 0.2*loss_fn_stab(raw_stab, y_test_stabi) + 0.6*loss_fn_latt(raw_latt, y_test_latsz) + 1.5*loss_fn_form(raw_form, y_test_eform)

    # Optional: Print loss every few epochs
    # if (total_validation_loss.item() > min_validation_loss+0.02) and (epoch > 300):
    #     break
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{train_epochs}], Loss: {total_loss_val.item():.4f}, Validation Loss: {total_validation_loss.item():.4f}')
    # min_validation_loss = min_validation_loss if (min_validation_loss < total_validation_loss.item()) else total_validation_loss.item()

Epoch [1000/5000], Loss: 0.0395, Validation Loss: 0.0482
Epoch [2000/5000], Loss: 0.0216, Validation Loss: 0.0605
Epoch [3000/5000], Loss: 0.0199, Validation Loss: 0.0884
Epoch [4000/5000], Loss: 0.0125, Validation Loss: 0.1060
Epoch [5000/5000], Loss: 0.0088, Validation Loss: 0.1197


In [5]:
my_cnn_net.eval()

with torch.no_grad():
    # raw_stab = my_cnn_net(x_test)
    raw_stab, raw_latt, raw_form = my_cnn_net(x_test)
    
    stability_preds = (raw_stab > 0.8).float()
    lattconst_preds = raw_latt*latt_sz_max
    eform_preds = raw_form*e_form_max
    # print(stability_preds, y_train_stabi)
    # print(lattconst_preds, y_train_latsz*latt_sz_max)
    # print(eform_preds, y_train_eform*e_form_max)
    
    error_stab = (stability_preds!=y_test_stabi)
    error_lattconst = (abs(lattconst_preds - y_test_latsz*latt_sz_max) > 0.3).float()
    error_eform = (abs(eform_preds - y_test_eform*e_form_max) > 0.1).float()
    total_data = y_test_stabi.shape[0]
    # print(f"Total Error Count For:\n  -Stability = {float(error_stab.sum())}")
    print(f"Total Error Count For:\n  -Stability = {float(error_stab.sum())}\n  -lattconst = {float(error_lattconst.sum())}\n  -e_formation = {float(error_eform.sum())}")
    # print(f"Total Accuracy For:\n  -Stability = {1-float(error_stab.sum())/total_data}")
    print(f"Total Accuracy For:\n  -Stability = {1-float(error_stab.sum())/total_data}\n  -lattconst = {1-float(error_lattconst.sum())/total_data}\n  -e_formation = {1-float(error_eform.sum())/total_data}")
    # print(f"Stability error for:\n{df['formula'][error_stab.numpy().reshape(-1)]}")
    

Total Error Count For:
  -Stability = 18.0
  -lattconst = 41.0
  -e_formation = 49.0
Total Accuracy For:
  -Stability = 0.9142857142857143
  -lattconst = 0.8047619047619048
  -e_formation = 0.7666666666666666
